## Tello 드론 Mission Pad — go / curve / jump 파이썬 코드 (중급)

---

## 📋 사전 준비
```bash
프로젝트 환경 만들기
  mkdir missionpad         : 프로젝트폴더생성
  cd missionpad            : 프로젝트폴더로 이동
  py --list                : 파이썬 버전 목록 확인
  py -3.14 -m venv venv    : 파이썬 가상환경 만들기
  venv\Scripts\activate    : 가상환경 시작( 종료는 : deactivate )
  pip install jupyterlab   : 주피터랩 패키지 설치
  pip install djitellopy   : DJI드론 API 설치
  jupyter lab --port=10888 : 주피터랩 시작
```

```
Mission Pad ID 규칙:
  mid 1~8 = m1~m8 (Tello SDK 기준)
  go/curve : 현재 위치 → 미션패드 기준 좌표로 이동
  jump     : 미션패드1 → 미션패드2 간 이동
```

---  


## 📊 명령어 파라미터 요약
<span style="position:left;display:inline-block;">
<table style="font-size:17px;">
    <tr align=center>
        <td>명령</td>
        <td>파라미터</td>
        <td>제한값</td>
    </tr>
    <tr>
        <td align=center>go</td>
        <td>x y z speed mid<br>미션패드 기준 좌표로 직선 이동</td>
        <td>xyz: -500~500<br>speed: 10~100</td>
    </tr>
    <tr>
        <td align=center>curve</td>
        <td>x1 y1 z1  x2 y2 z2  speed mid<br>경유점(x1y1z1) → 도착점(x2y2z2) 곡선 비행</td>
        <td>xyz: -500~500<br>speed: 10~60</td>
    </tr>
    <tr>
        <td align=center>jump</td>
        <td>x y z speed yaw mid1 mid2<br>미션패드1 → 미션패드2 간 이동</td>
        <td>xyz: -500~500<br>yaw: 0~360</td>
    </tr>
    <tr colspan=3><td>○ mid (Mission Pad ID) : 1~8 (m1~m8)<br>
○ 단위 : 좌표(cm), 속도(cm/s), 각도(도)</td></tr>
</table>
</span>

---


## 🟢 중급 (Intermediate) — djitellopy 라이브러리 활용


In [ ]:
from djitellopy import Tello
import time

def run_mission():
    drone = Tello()

    # ── 연결 및 초기화 ──────────────────────────────
    drone.connect()
    print(f"배터리: {drone.get_battery()}%")

    # 미션패드 감지 활성화
    # 방향: 0=아래, 1=앞, 2=아래+앞
    drone.enable_mission_pads()
    drone.set_mission_pad_detection_direction(0)  # 아래 방향 감지
    time.sleep(1)

    drone.takeoff()
    time.sleep(3)

    # ── 미션패드 감지 확인 ──────────────────────────
    mid = drone.get_mission_pad_id()
    print(f"감지된 미션패드: {mid}")

    # ── go 명령 ────────────────────────────────────
    # 미션패드 m1 기준 (x=0, y=0, z=80) 위치로 이동
    if mid == 1:
        print(">>> go 명령 실행")
        drone.go_xyz_speed_mid(
            x=0, y=0, z=80,    # 미션패드 기준 좌표 (cm)
            speed=30,           # 속도 (10~100 cm/s)
            mid=1               # 미션패드 ID
        )
        time.sleep(4)

    # ── curve 명령 ─────────────────────────────────
    # 경유점 → 도착점 곡선 비행
    print(">>> curve 명령 실행")
    drone.curve_xyz_speed_mid(
        x1=50, y1=0,  z1=80,   # 경유점 (arc point)
        x2=50, y2=50, z2=80,   # 도착점 (end point)
        speed=30,               # 속도 (10~60 cm/s)
        mid=1
    )
    time.sleep(5)

    # ── jump 명령 ──────────────────────────────────
    # 미션패드1 → 미션패드2 이동
"""
    print(">>> jump 명령 실행")
    drone.go_xyz_speed_yaw_mid(
        x=0, y=0, z=80,        # mid2 기준 도착 좌표
        speed=30,
        yaw=0,                  # 도착 후 회전각 (0~360도)
        mid1=1,                 # 출발 미션패드
        mid2=2                  # 도착 미션패드
    )
    time.sleep(5)
"""
    # ── 착륙 ───────────────────────────────────────
    drone.land()
    drone.disable_mission_pads()
    drone.end()

if __name__ == '__main__':
    run_mission()